# zhjudge：在 Colab / Kaggle 的 GPU 上训练 / Train on a Colab or Kaggle GPU

一键跑完整流程：建数据 → 训练 → 校准 → 评估 → 导出 Laya 格式检查点。
Runs the whole pipeline: build data → train → calibrate → evaluate → export a Laya-format checkpoint.

> 独立开源项目，与 TypeSafe（Jev）和 Convai（Laya）没有任何隶属、合作或背书关系；提及其名称只为说明模型类别（Jev 式类型化决策）和 Laya 检查点格式。不调用 Jev 的接口，也从不用它的输出训练。
> Independent open-source project, not affiliated with or endorsed by TypeSafe (Jev) or Convai (Laya); their names only identify the model category (Jev-style typed decisions) and the Laya checkpoint format. Jev's API is never called and its outputs are never trained on.

**开始前 / Before you start**
1. Colab：菜单「代码执行程序 → 更改运行时类型 → T4 GPU」。Kaggle：Settings → Accelerator = GPU T4，并打开 Internet。
   Colab: Runtime → Change runtime type → T4 GPU. Kaggle: Settings → Accelerator = GPU T4, Internet on.
2. 训练数据：`configs/datasets.yaml` 已按许可审计打开 24 个来源（见 DATA_LICENSES.md）。要 Apache-2.0 权重，把下面的 `LICENSE_PROFILE` 改成 `permissive_strict`。
   Training data: `configs/datasets.yaml` enables the 24 sources the licence audit allows (DATA_LICENSES.md). For Apache-2.0 weights set `LICENSE_PROFILE = "permissive_strict"` below.
3. 时间：Kaggle 单张 T4 上实测每个 epoch 约 1.4 小时（v0，约 22 万条训练样本，2 个 epoch 训练 2.8 小时；v1b 训练 3.2 小时）；日志里的 `ETA` 和结束后 `train_stats.json` 的
   `hours_per_epoch_estimate` 按实测吞吐计算。免费 Colab 可能中途断开：打开 `USE_DRIVE` 把 runs 放到 Google Drive，重连后从上到下重新运行全部单元格即可从断点继续。
   Kaggle 的新会话从空目录开始：把上一版本（Save Version）的输出作为 Input 挂上，并把 `RESUME_FROM` 设为其中的 runs 目录。
   Measured on a Kaggle T4: about 1.4 h per epoch (v0, ~220k training records, 2.8 h for 2 epochs; v1b 3.2 h); the log's `ETA` and
   `hours_per_epoch_estimate` in `train_stats.json` come from the measured throughput. If a Colab session drops, keep runs on
   Google Drive (`USE_DRIVE`), reconnect and rerun all cells from the top: training resumes from the last checkpoint. A new Kaggle session starts empty: attach the previous
   version's output as an input and set `RESUME_FROM` to the runs directory inside it.
4. 仓库是公开的，不需要 GitHub 令牌。只有用私有仓库（例如你用 GitHub 的 Import repository 建的私有副本）时：把 `REPO_URL` 改成它，Colab 从 GitHub 打开笔记本时勾选 "Include private repos"，并把下面的 `PRIVATE_REPO` 设为 `True`（需要一个只读 fine-grained token，只在内存里）。
   The repo is public: no GitHub token needed. Only for a private repo (e.g. a private copy made with GitHub's Import repository): set `REPO_URL` to it, tick "Include private repos" when Colab opens the notebook from GitHub and set `PRIVATE_REPO = True` (asks for a read-only fine-grained token, kept in memory only).

In [ ]:
# 1) 查看 GPU / check the GPU
!nvidia-smi || echo "no NVIDIA GPU visible: switch the runtime to a GPU first"

In [ ]:
# 2) 设置 / settings
import os

REPO_URL = "https://github.com/TerryG907/zhjudge"   # 你的仓库地址 / your repo
BRANCH = "main"
PRIVATE_REPO = False                               # 公开仓库不需要令牌。True = 私有仓库：Kaggle 从 Secrets 读取 GITHUB_TOKEN，否则运行时粘贴（不写入文件）/ True only for a private repo: Kaggle Secret GITHUB_TOKEN, else pasted at runtime
CONFIG = "configs/train_base.yaml"                 # configs/train_smoke.yaml = 几分钟的流程自检 / minutes-long pipeline check
RUN_NAME = ""                                      # 留空 = 用配置里的 run.name / empty = run.name from the config
USE_DRIVE = True                                   # Colab：把 runs/ 存到 Google Drive，断线可续训 / keep runs on Drive to resume
MAX_HOURS = 0                                      # >0：到时保存并退出（Kaggle 建议 11）/ save + exit after this many hours
LICENSE_PROFILE = "default"                        # default（CC BY-SA 4.0 权重）| permissive_strict（Apache-2.0 权重）
RESUME_FROM = ""                                   # 从这里复制 runs/ 再开始（Kaggle：上一版本输出里的 runs，先 !ls /kaggle/input 找路径）
                                                   # copy runs/ from here first (Kaggle: previous version output, e.g. /kaggle/input/<name>/zhjudge/runs)

ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
REPO_DIR = f"{WORK}/zhjudge"
if ON_KAGGLE:
    # keep the ~6 GB virtualenv out of /kaggle/working (20 GB limit, saved as notebook output)
    os.environ["UV_PROJECT_ENVIRONMENT"] = "/tmp/zhjudge-venv"
if ON_KAGGLE and not MAX_HOURS:
    MAX_HOURS = 11                                 # Kaggle 单次会话上限 12 小时，11 小时保存并退出 / Kaggle sessions cap at 12 h
print("Kaggle" if ON_KAGGLE else "Colab", REPO_DIR, "MAX_HOURS =", MAX_HOURS)

In [ ]:
# 3) （可选）挂载 Google Drive 保存 runs / (optional) mount Google Drive for runs/
if USE_DRIVE and not ON_KAGGLE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["ZHJUDGE_RUNS_DIR"] = "/content/drive/MyDrive/zhjudge/runs"
    os.makedirs(os.environ["ZHJUDGE_RUNS_DIR"], exist_ok=True)
if RUN_NAME:
    os.environ["RUN_NAME"] = RUN_NAME
print("runs dir:", os.environ.get("ZHJUDGE_RUNS_DIR", f"{REPO_DIR}/runs"))

In [ ]:
# 4) 获取代码并安装 uv / get the code and install uv
import subprocess
git_env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
if PRIVATE_REPO:
    # 只读的 fine-grained token（Contents: read-only）。通过环境变量只传给下面这两条 git 命令，
    # 不出现在命令行，也不写入 .git/config（Kaggle 会把 /kaggle/working 存成输出）。
    # Read-only fine-grained token, passed to these git calls through the environment only:
    # never on a command line and never written to .git/config (Kaggle saves /kaggle/working as output).
    import base64
    _tok = ""
    if ON_KAGGLE:
        # Kaggle：Add-ons → Secrets 里添加名为 GITHUB_TOKEN 的密钥并勾选本笔记本。后台 Save Version 运行无法弹框，只能用这种方式。
        # Kaggle: Add-ons → Secrets → add GITHUB_TOKEN and attach it to this notebook (required for background "Save Version" runs).
        try:
            from kaggle_secrets import UserSecretsClient
            _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception as e:
            print("Kaggle secret GITHUB_TOKEN not available:", type(e).__name__)
    _tok = _tok or os.environ.pop("GITHUB_TOKEN", "")
    if not _tok:
        from getpass import getpass
        _tok = getpass("GitHub token, read-only (not stored): ")
    _auth = base64.b64encode(("x-access-token:" + _tok).encode()).decode()
    del _tok
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0="AUTHORIZATION: basic " + _auth)
    del _auth
try:
    if os.path.isdir(f"{REPO_DIR}/.git"):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True, env=git_env)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True, env=git_env)
finally:
    del git_env
!pip install -q uv
!uv --version

In [ ]:
# 5) 跑完整流程（可重复运行：已完成的训练会跳过，未完成的会续训）
#    Run the pipeline (safe to rerun: finished training is skipped, unfinished training resumes)
import shutil
os.environ["CONFIG"] = CONFIG
runs_dir = os.environ.get("ZHJUDGE_RUNS_DIR", f"{REPO_DIR}/runs")
if RESUME_FROM and not (os.path.isdir(runs_dir) and os.listdir(runs_dir)):
    # only into an empty runs dir, so rerunning this cell never replaces newer progress with the older copy
    shutil.copytree(RESUME_FROM, runs_dir, dirs_exist_ok=True)  # resumable ckpt/last.pt is ~4 GB for mmBERT-base
    print("copied", RESUME_FROM, "->", runs_dir)
extra = f"--set train.max_hours={MAX_HOURS}" if MAX_HOURS else ""
if LICENSE_PROFILE != "default":
    extra += f" --set data.license_profile={LICENSE_PROFILE} --set export.license=apache-2.0"
!cd {REPO_DIR} && bash scripts/run_all.sh {extra}

In [ ]:
# 6) 查看评估结果 / show the evaluation
import glob
runs_dir = os.environ.get("ZHJUDGE_RUNS_DIR", f"{REPO_DIR}/runs")
for p in sorted(glob.glob(f"{runs_dir}/*/eval.md")):
    print("=" * 20, p)
    print(open(p, encoding="utf-8").read())

In [ ]:
# 7) （可选）把导出的检查点复制到 Google Drive / (optional) copy the export to Google Drive
#    若 runs 已在 Drive 上则无需复制 / not needed when runs/ already lives on Drive
import glob, shutil
runs_dir = os.environ.get("ZHJUDGE_RUNS_DIR", f"{REPO_DIR}/runs")
if not ON_KAGGLE and not os.environ.get("ZHJUDGE_RUNS_DIR"):
    from google.colab import drive
    drive.mount("/content/drive")
    for d in glob.glob(f"{runs_dir}/*/export/*"):
        dst = "/content/drive/MyDrive/zhjudge/exports/" + os.path.basename(d)
        shutil.copytree(d, dst, dirs_exist_ok=True)
        print("copied", d, "->", dst)
else:
    print("exports are in", runs_dir, "(on Kaggle: download from the Output tab)")

In [ ]:
# 8) （可选）上传到 Hugging Face（默认私有仓库）。Token 只在本次运行的内存里，不写入文件或笔记本。
#    (optional) push to Hugging Face (private repo by default). The token stays in memory for this call only.
HF_REPO = ""   # 例如 / e.g. "your-name/zhjudge-mmbert-base"
lic_sets = (f"--set data.license_profile={LICENSE_PROFILE} --set export.license=apache-2.0"
            if LICENSE_PROFILE != "default" else "")
if HF_REPO:
    from getpass import getpass
    os.environ["HF_TOKEN"] = getpass("Hugging Face write token (not stored): ")
    try:
        !cd {REPO_DIR} && uv run --no-sync zhjudge export --config {CONFIG} {lic_sets} --push-to {HF_REPO}
    finally:
        os.environ.pop("HF_TOKEN", None)

### 其他云 GPU（AutoDL / RunPod 等）/ Other cloud GPUs

```bash
git clone https://github.com/TerryG907/zhjudge && cd zhjudge
pip install uv            # 或 / or: curl -LsSf https://astral.sh/uv/install.sh | sh
CONFIG=configs/train_base.yaml bash scripts/run_all.sh
```
断线后重新执行同一命令即可续训（runs/ 在同一台机器的磁盘上）。Rerun the same command after a disconnect to resume (runs/ stays on the machine's disk).